# Ejercicio 1: Análisis Sociodemográfico de Ingresos

El script debe contener una sección de preparación de los datos enfocada en características sociodemográficas:

1. Cargar el dataset Adult. Consultar en https://archive.ics.uci.edu/dataset/2/adult
2. Identificar y analizar las variables sociodemográficas clave: income, age, sex, native-country
3. Realizar un análisis descriptivo enfocado en la distribución de ingresos por características sociodemográficas
4. Decidir qué tratamiento darle a valores faltantes.
5. Preparar los datos para análisis de clustering con enfoque sociodemográfico

In [1]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd

# Cargar el dataset "Adult" desde archivos locales
df = cargar_dataset_adult()

df


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States,<=50K
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K


## Análisis de Variables Sociodemográficas Clave

**Variables de interés para el análisis de ingresos:**
- **income**: Variable objetivo (>$50K o ≤$50K)
- **age**: Edad del individuo
- **sex**: Género (Male/Female)
- **native-country**: País de origen

**Otras variables relevantes para clustering:**
- Variables numéricas: age, education-num, capital-gain, capital-loss, hours-per-week
- Variables categóricas: workclass, education, marital-status, occupation, relationship, race, sex, native-country

In [2]:
from IPython.display import display
import pandas as pd

# Nota: fnlwgt (final weight) NO se incluye en el análisis porque es un peso de muestreo del censo,
# no una característica descriptiva del individuo. Explicado en "notebook.md" de la carpeta "/Docs"

data = {'Variable': ['age', 'workclass', 'education', 'education-num', 'marital-status', 'occupation', 
                     'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 
                     'native-country', 'income'],
        'Tipo': ['cuantitativa discreta', 'cualitativa nominal', 'cualitativa ordinal', 'cuantitativa discreta', 
                 'cualitativa nominal', 'cualitativa nominal', 'cualitativa nominal', 'cualitativa nominal', 
                 'cualitativa nominal', 'cuantitativa continua', 'cuantitativa continua', 'cuantitativa discreta',
                 'cualitativa nominal', 'cuantitativa continua']}

variable_type_df = pd.DataFrame(data)
display(variable_type_df)

,Variable,Tipo
0,age,cuantitativa discreta
1,workclass,cualitativa nominal
2,education,cualitativa ordinal
3,education-num,cuantitativa discreta
4,marital-status,cualitativa nominal
5,occupation,cualitativa nominal
6,relationship,cualitativa nominal
7,race,cualitativa nominal
8,sex,cualitativa nominal
9,capital-gain,cuantitativa continua


## Análisis de valores faltantes

In [3]:
"""
Análisis de valores faltantes
"""
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
from IPython.display import display
import pandas as pd
import numpy as np

# Cargar el dataset desde archivos locales
df = cargar_dataset_adult()

print("Análisis de Valores Faltantes en el Dataset 'Adult'")
print("-" * 50)

# 1. Calcular el número de valores faltantes por columna
conteo_faltantes = df.isnull().sum()

# 2. Calcular el total de filas del DataFrame
total_filas = len(df)

# 3. Calcular el porcentaje de valores faltantes
porcentaje_faltantes = (conteo_faltantes / total_filas) * 100

# 4. Crear un DataFrame para mostrar los resultados de forma clara
resumen_faltantes = pd.DataFrame({
    'Conteo Faltantes': conteo_faltantes,
    'Porcentaje (%)': porcentaje_faltantes
})

# 5. Ordenar la tabla para ver las columnas más problemáticas primero
resumen_faltantes = resumen_faltantes.sort_values(by='Porcentaje (%)', ascending=False)

# 6. Filtrar para mostrar solo las columnas que SÍ tienen valores faltantes
resumen_faltantes = resumen_faltantes[resumen_faltantes['Conteo Faltantes'] > 0]

# 7. Imprimir el resultado final
if resumen_faltantes.empty:
    print("¡Felicidades! No se encontraron valores faltantes en el dataset.")
else:
    print(f"Resultados sobre un total de {total_filas} filas:\n")
    display(resumen_faltantes)

Análisis de Valores Faltantes en el Dataset 'Adult'
--------------------------------------------------
Resultados sobre un total de 48842 filas:



,Conteo Faltantes,Porcentaje (%)
occupation,2809,5.751198
workclass,2799,5.730724
native-country,857,1.754637


## Tratamiento de valores faltantes

Ya que los datos faltantes son mayores a un 5%, los vamos a tratar. Tenemos dos formas de hacerlo:

- **Imputación por constante**: Reemplazar "?" con una constante como "Desconocido"
- **Imputación por moda**: Reemplazar por la moda de esa columna

**Nos pareció mejor opción imputar por moda**


In [4]:
# =================================================================
# Imputación con la Moda
# =================================================================

from IPython.display import display
from sklearn.impute import SimpleImputer
import pandas as pd

print("--- Imputación con la Moda ---")

# Creamos una copia del DataFrame original para no modificarlo
df_imputado_moda = df.copy()

# 2. Definimos las columnas que vamos a rellenar
columnas_a_imputar = ['workclass', 'occupation', 'native-country']

# 3. Creamos el imputador con la estrategia 'most_frequent'
imputador_moda = SimpleImputer(strategy='most_frequent')

# 4. Aplicamos el imputador
df_imputado_moda[columnas_a_imputar] = imputador_moda.fit_transform(df_imputado_moda[columnas_a_imputar])

print("\nMostrando el DataFrame imputado con la moda:")

display(df_imputado_moda)

--- Imputación con la Moda ---

Mostrando el DataFrame imputado con la moda:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K
48838,64,Private,321403,HS-grad,9,Widowed,Prof-specialty,Other-relative,Black,Male,0,0,40,United-States,<=50K
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K


## Análisis Sociodemográfico de Ingresos


In [5]:
# Análisis sociodemográfico enfocado en ingresos
import matplotlib.pyplot as plt
import seaborn as sns

print("="*70)
print("ANÁLISIS SOCIODEMOGRÁFICO DE INGRESOS")
print("="*70)

# 1. Distribución general de ingresos
print("\n1. DISTRIBUCIÓN DE INGRESOS:")
income_dist = df['income'].value_counts()
print(income_dist)
print(f"\nPorcentaje de ingresos altos (>$50K): {(income_dist['>50K'] / len(df)) * 100:.2f}%")

# 2. Análisis por género
print("\n" + "="*50)
print("2. ANÁLISIS POR GÉNERO:")
print("="*50)

gender_income = pd.crosstab(df['sex'], df['income'], margins=True)
print("\nDistribución de ingresos por género:")
print(gender_income)

# Calcular porcentajes por género
gender_pct = pd.crosstab(df['sex'], df['income'], normalize='index') * 100
print("\nPorcentajes por género:")
print(gender_pct.round(2))

# 3. Análisis por edad
print("\n" + "="*50)
print("3. ANÁLISIS POR EDAD:")
print("="*50)

# Crear grupos de edad
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100], 
                        labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

age_income = pd.crosstab(df['age_group'], df['income'], margins=True)
print("\nDistribución de ingresos por grupo de edad:")
print(age_income)

age_pct = pd.crosstab(df['age_group'], df['income'], normalize='index') * 100
print("\nPorcentajes por grupo de edad:")
print(age_pct.round(2))



# 5. Estadísticas descriptivas por ingresos
print("\n" + "="*50)
print("5. ESTADÍSTICAS DESCRIPTIVAS POR INGRESOS:")
print("="*50)

print("\nEdad promedio por nivel de ingresos:")
age_by_income = df.groupby('income')['age'].agg(['mean', 'median', 'std'])
print(age_by_income.round(2))

print("\nDistribución de género por nivel de ingresos:")
print(gender_pct.round(2))

# 6. Resumen para clustering
print("\n" + "="*50)
print("6. RESUMEN PARA ANÁLISIS DE CLUSTERING:")
print("="*50)

print("\nVariables sociodemográficas clave identificadas:")
print("- income: Variable objetivo (binaria)")
print("- age: Variable numérica continua")
print("- sex: Variable categórica binaria")

print(f"\nTamaño del dataset: {len(df):,} observaciones")
print(f"Variables totales: {len(df.columns)}")

ANÁLISIS SOCIODEMOGRÁFICO DE INGRESOS

1. DISTRIBUCIÓN DE INGRESOS:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Porcentaje de ingresos altos (>$50K): 23.93%

2. ANÁLISIS POR GÉNERO:

Distribución de ingresos por género:
income  <=50K   >50K    All
sex                        
Female  14423   1769  16192
Male    22732   9918  32650
All     37155  11687  48842

Porcentajes por género:
income  <=50K   >50K
sex                 
Female  89.07  10.93
Male    69.62  30.38

3. ANÁLISIS POR EDAD:

Distribución de ingresos por grupo de edad:
income     <=50K   >50K    All
age_group                     
18-25       9458    169   9627
26-35      10340   2379  12719
36-45       7821   4131  11952
46-55       5041   3255   8296
56-65       3049   1396   4445
65+         1446    357   1803
All        37155  11687  48842

Porcentajes por grupo de edad:
income     <=50K   >50K
age_group              
18-25      98.24   1.76
26-35      81.30  18.70
36-45      65.44  34.56
46-55      6

## Análisis Descriptivo


Disponible en la carpeta `/Docs` en el archivo `ejercicio1.md`